[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/srini2k03/jobpost-extractor-qlora/blob/main/notebooks/02_baselines.ipynb)

# 02 · Prompting baselines (zero-/few-shot, free vs guided JSON)

Runs the base models with the **same system prompt and schema** used for fine-tuning: Qwen2.5-1.5B-Instruct zero-shot and 3-shot, the optional Llama-3.2-1B-Instruct 3-shot, and the **Qwen2.5-7B-Instruct 4-bit 3-shot upper bound**. Each runs with free decoding and with **guided JSON decoding** (lm-format-enforcer), to show what decoding constraints alone buy. Same code as `make local-baselines` (`jobx.pipeline.run_family`); finished rows are skipped.

**How to run**
1. Runtime → Change runtime type → **T4 GPU**.
2. Secrets (key icon in the left sidebar), each with *Notebook access* switched on: `HF_TOKEN`, `HF_USERNAME`.
   `HF_USERNAME` is your Hugging Face account name; `HF_TOKEN` needs *write* access.
3. Runtime → **Run all**. Expected runtime: **~2–2.5 h in total; each section saves and pushes its own results, so you can split it across sessions with the `RUN_*` flags**. Cells after the setup are safe to re-run.

**Outputs:** `results/predictions/<approach>__<split>.jsonl` + `.meta.json` and `results/metrics/<approach>__<split>.json` for every row, pushed to `<HF_USERNAME>/jobx-artifacts`.

All code lives in `src/jobx` of the GitHub repo; this notebook only orchestrates it.

In [ ]:
# Setup: clone the repo (code + data), install pinned dependencies, install jobx itself.
import os, sys
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/srini2k03/jobpost-extractor-qlora.git"
if IN_COLAB:
    if not os.path.exists("/content/jobpost-extractor-qlora"):
        !git clone --depth 1 {REPO_URL} /content/jobpost-extractor-qlora
    %cd /content/jobpost-extractor-qlora
    !pip install -q transformers==4.56.2 peft==0.17.1 trl==0.23.1 bitsandbytes==0.48.1 accelerate==1.10.1 datasets==4.1.1 huggingface_hub==0.35.3 wandb==0.22.2 lm-format-enforcer==0.11.3
    !pip install -q -e .
else:
    # running locally from a checkout: notebooks/ -> repo root
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU'
print(torch.__version__, torch.cuda.get_device_name(0))


In [ ]:
# Secrets -> environment variables (read by jobx.hub and wandb).
import os
for key in ("HF_TOKEN", "HF_USERNAME", "WANDB_API_KEY"):
    if not os.environ.get(key):
        try:
            from google.colab import userdata
            os.environ[key] = userdata.get(key)
        except Exception:
            print(f"secret {key} not available")
from jobx.hub import hf_username, repo_id
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("Hub account:", hf_username(), "| artifacts repo:", repo_id("artifacts"))

## Settings
Guided decoding runs one posting at a time, so it uses a fixed, evenly spaced subset (`guided_max` in `configs/gpu/colab_t4.yaml`) that is identical for every approach; the table reports `n`.

In [ ]:
RUN_QWEN_1_5B = True
RUN_LLAMA_1B = False   # gated model: accept the licence on its HF page first
RUN_QWEN_7B = True
from jobx import prompts
from jobx.baselines.runs import available_test_splits
from jobx.data.build import read_jsonl
from jobx.pipeline import load_profile, run_family
PROFILE = load_profile('colab_t4')
SPLITS = available_test_splits()
SHOTS = prompts.select_few_shot(read_jsonl('data/processed/train.jsonl'), k=3)
print('splits:', SPLITS, '| shots:', [s['id'] for s in SHOTS])

## Qwen2.5-1.5B-Instruct (16-bit), ~45 min

In [ ]:
if RUN_QWEN_1_5B:
    run_family('qwen1.5b', PROFILE, splits=SPLITS, shots=SHOTS, push=True)

## Llama-3.2-1B-Instruct (16-bit, optional), ~30 min

In [ ]:
if RUN_LLAMA_1B:
    run_family('llama1b', PROFILE, splits=SPLITS, shots=SHOTS, push=True)

## Upper bound: Qwen2.5-7B-Instruct 3-shot, 4-bit NF4, ~75 min

In [ ]:
if RUN_QWEN_7B:
    run_family('qwen7b', PROFILE, splits=SPLITS, shots=SHOTS, push=True)

## Summary

In [ ]:
from jobx.eval.results_table import build_tables
from IPython.display import Markdown
Markdown(build_tables('.')[0])